## Document Translation


For Azure Document Translation to work properly, the service needs to be able to read from your source container and write to your target container. Here's how permissions work:

1. **Source Container**: Needs `read` and `list` permissions so the translation service can access the files
2. **Target Container**: Needs `write`, `create`, `read`, and `list` permissions for the service to write translated documents

The most common approaches to grant these permissions are:

- **SAS (Shared Access Signature) tokens**: Temporary, limited-privilege access to your storage
- **Azure AD integration**: Using managed identities (most secure for production)

This notebook demonstrates both the SAS token approach and public access approach.

In [ ]:
%pip install openai
%pip install python-dotenv
%pip install azure-ai-translation-document==1.0.0
%pip install azure-core
# Install Azure Blob Storage libraries
%pip install azure-storage-blob
# Install Azure Identity for managed identity support
%pip install azure-identity

In [ ]:
# Import required libraries for Azure Document Translation
from azure.ai.translation.document import DocumentTranslationClient, TranslationTarget, DocumentTranslationInput
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv(dotenv_path='../keys.env')
translator_endpoint = os.getenv('AZURE_TRANSLATOR_DOCUMENT_ENDPOINT')
translator_key = os.getenv('AZURE_TRANSLATOR_KEY')
storage_account_name = os.getenv('AZURE_STORAGE_ACCOUNT_NAME')
storage_account_key = os.getenv('AZURE_STORAGE_ACCOUNT_KEY')
sas_output = os.getenv('AZURE_STORAGE_SAS_OUTPUT')

# Verify required environment variables
if not all([translator_endpoint, translator_key, storage_account_name, storage_account_key]):
    raise ValueError("Missing required environment variables. Check your .env file.")

# Define container names and file path
source_container_name = "source-documents"
target_container_name = "translated-documents"
local_md_file = '../data/hachette/ScreenTime/Screen Time - Lisa Guernsey_full_text.md'
targetLanguage = 'ga'  # French fr -  ga for irish Supported languages https://learn.microsoft.com/en-us/azure/ai-services/translator/language-support 

# Create BlobServiceClient for file operations (not for SAS generation)
blob_service_client = BlobServiceClient(
    account_url=f"https://{storage_account_name}.blob.core.windows.net", 
    credential=storage_account_key
)

# Upload the file to the source container
blob_name = os.path.basename(local_md_file)
blob_client = blob_service_client.get_blob_client(container=source_container_name, blob=blob_name)

# Ensure the containers exist with private access
for container_name in [source_container_name, target_container_name]:
    try:
        container_client = blob_service_client.get_container_client(container_name)
        container_client.get_container_properties()
        print(f"Container {container_name} already exists")
    except Exception:
        print(f"Creating container: {container_name}")
        blob_service_client.create_container(container_name, public_access=None)

# Upload the document
with open(local_md_file, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)
    print(f"Uploaded {blob_name} to {source_container_name} container")

# Use pre-generated SAS URLs (these would be provided externally)
# For a production environment, these URLs might be retrieved from a secure service or Azure Key Vault
source_sas_url = os.getenv('SOURCE_CONTAINER_SAS_URL')
target_sas_url = os.getenv('TARGET_CONTAINER_SAS_URL')
if not source_sas_url or not target_sas_url:
    # Replace these placeholder URLs with your actual pre-generated SAS URLs
    source_sas_url = f"https://{storage_account_name}.blob.core.windows.net/{source_container_name}?{sas_output}"
    target_sas_url = f"https://{storage_account_name}.blob.core.windows.net/{target_container_name}?{sas_output}"

# Create the Document Translation client
client = DocumentTranslationClient(translator_endpoint, AzureKeyCredential(translator_key))

# Start translation using the provided SAS URLs with custom options
print("Starting document translation with custom filename format...")
print("Waiting for translation to complete (this may take time)...")
try:
    # Use advanced options for customization
    poller = client.begin_translation(
        source_url=source_sas_url,
        target_url=target_sas_url,
        target_language=targetLanguage
    )
    
    result = poller.result()
    
    # Display translation details
    print(f'Status: {poller.status()}')
    print(f'Created on: {poller.details.created_on}')
    print(f'Last updated: {poller.details.last_updated_on}')
    print(f'Total documents: {poller.details.documents_total_count}')
    
    print('\nDocument summary:')
    print(f'{poller.details.documents_failed_count} failed')
    print(f'{poller.details.documents_succeeded_count} succeeded')

    
    for document in result:
        print(f'\nDocument ID: {document.id}')
        print(f'Document status: {document.status}')
        if document.status == 'Succeeded':
            print(f'Source document URL: {document.source_document_url}')
            print(f'Translated document URL: {document.translated_document_url}')
            print(f'Translated to: {document.translated_to}')

        else:
            print(f'Error code: {document.error.code}, Message: {document.error.message}')

except Exception as e:
    print(f"Error during translation: {str(e)}")
    # Additional error information for troubleshooting
    print("Check your SAS tokens and ensure they have the correct permissions:")
    print("- Source container: read, list")
    print("- Target container: read, write, list, create")



Container source-documents already exists
Container translated-documents already exists
Uploaded Screen Time - Lisa Guernsey_full_text.md to source-documents container
Starting document translation with custom filename format...
Waiting for translation to complete (this may take time)...
Status: Succeeded
Created on: 2025-04-09 14:13:41.110296+00:00
Last updated: 2025-04-09 14:13:58.478132+00:00
Total documents: 1

Document summary:
0 failed
1 succeeded

Document ID: 0181362a-0000-0000-0000-000000000000
Document status: Succeeded
Source document URL: https://doctranslateaudiogen.blob.core.windows.net:443/source-documents/Screen%20Time%20-%20Lisa%20Guernsey_full_text.md
Translated document URL: https://doctranslateaudiogen.blob.core.windows.net:443/translated-documents/Screen%20Time%20-%20Lisa%20Guernsey_full_text.md
Translated to: ga
